# Extract Bio-Assay Data from Pubchem Database for all target `Bacteria`

In [30]:
import os
from chembl_webresource_client.new_client import new_client
from tqdm import tqdm
import os
import time
import random
from typing import Dict, List, Optional

import requests
import pandas as pd
import numpy as np
from rdkit import Chem
import matplotlib.pyplot as plt
from pathlib import Path

### Read paths

In [31]:
# Base directory path
dataDir = '/mnt/data.ese/nfs/users/sghosh6/DTRA_project/MACAW/DrugDesignData/'
modelBuildingDataDir = os.path.join(dataDir, 'modelBuildingData/')

In [32]:
PubchemDataDir = os.path.join(dataDir, 'PubChem')
BacillusAnthracis_PubchemDataDir = os.path.join(PubchemDataDir, 'PubChem_BacillusAnthracis')
YersiniaPestis_PubchemDataDir = os.path.join(PubchemDataDir, 'PubChem_Lassa')

In [33]:
def loadAllCsvsToOneDF(BacteriaPubchemDataDir: str | Path) -> pd.DataFrame:
    BacteriaPubchemDataDir = Path(BacteriaPubchemDataDir)

    DFList = []
    for csvPath in sorted(BacteriaPubchemDataDir.glob("*.csv")):
        try:
            DF = pd.read_csv(csvPath, dtype=str, low_memory=False)
            DF.insert(0, "sourceFile", csvPath.name)
            DFList.append(DF)
        except Exception as e:
            print(f"[SKIP] {csvPath.name}: {e}")

    return pd.concat(DFList, ignore_index=True, sort=False) if DFList else pd.DataFrame()


# ---- Provide the directory for each Bacteria ----
BacteriaToDirDict = {
    "Bacillus_anthracis":   BacillusAnthracis_PubchemDataDir,
    "Yersinia_pestis":   YersiniaPestis_PubchemDataDir,
}

DFList = []
for BacteriaName, BacteriaDir in BacteriaToDirDict.items():
    DF = loadAllCsvsToOneDF(BacteriaDir)
    DF["Bacteria"] = BacteriaName
    DFList.append(DF)

allBacteria_Pubchem = pd.concat(DFList, ignore_index=True, sort=False)
allBacteria_Pubchem

[SKIP] AID_588467.csv: No columns to parse from file
[SKIP] AID_588469.csv: No columns to parse from file
[SKIP] AID_588470.csv: No columns to parse from file


/tmp/ipykernel_2704287/3952925494.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  DF["Bacteria"] = BacteriaName


[SKIP] AID_463114.csv: No columns to parse from file
[SKIP] AID_540249.csv: No columns to parse from file


,sourceFile,PUBCHEM_RESULT_TAG,PUBCHEM_SID,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,PUBCHEM_ACTIVITY_URL,PUBCHEM_ASSAYDATA_COMMENT,Standard Type,...,Activity at 0.076 uM,Activity at 0.219 uM,Activity at 0.631 uM,Activity at 1.728 uM,Activity at 3.886 uM,Activity at 8.587 uM,Activity at 17.80 uM,Activity at 49.20 uM,Activity at 107.3 uM,Activity at 231.0 uM
0,AID_1059137.csv,RESULT_TYPE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,STRING,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AID_1059137.csv,RESULT_DESCR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Standardized activity type (e.g. IC50 rather t...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AID_1059137.csv,1,103163904,2764,C1CC1N2C=C(C(=O)C3=CC(=C(C=C32)N4CCNCC4)F)C(=O)O,Unspecified,NaN,NaN,NaN,MIC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AID_1059137.csv,2,194138621,72736352,COC1CNC(=NC1)C2=CC3=C(C=C2)C=C(N3)C4=CC=C(C=C4...,Unspecified,NaN,NaN,NaN,MIC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AID_1059137.csv,3,194145553,72735775,C1CNC(=NC1)C2=CC3=C(C=C2)C=C(N3)CCC4=CC5=C(N4)...,Unspecified,NaN,NaN,NaN,MIC,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2005892,AID_775806.csv,36,174524730,135907423,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC=...,Active,NaN,NaN,NaN,EC50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005893,AID_775806.csv,37,174524731,73356996,CC1=CC(=CC=C1)/C(=N/NC(=O)[C@@H]2C[C@H]2C3=CC=...,Active,NaN,NaN,NaN,EC50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005894,AID_775806.csv,38,174524732,9695967,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC(...,Active,NaN,NaN,NaN,EC50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2005895,AID_775806.csv,39,174524733,25753841,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC=...,Active,NaN,NaN,NaN,EC50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
allBacteria_Pubchem = allBacteria_Pubchem[['PUBCHEM_CID', 'PUBCHEM_EXT_DATASOURCE_SMILES', 'PubChem Standard Value', 'Standard Type',
                                      'Standard Relation', 'Standard Value', 'Standard Units', 'Bacteria']]
allBacteria_Pubchem = allBacteria_Pubchem.dropna(subset=["PUBCHEM_EXT_DATASOURCE_SMILES"])
allBacteria_Pubchem

,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PubChem Standard Value,Standard Type,Standard Relation,Standard Value,Standard Units,Bacteria
2,2764,C1CC1N2C=C(C(=O)C3=CC(=C(C=C32)N4CCNCC4)F)C(=O)O,NaN,MIC,=,0.08,ug.mL-1,Bacillus_anthracis
3,72736352,COC1CNC(=NC1)C2=CC3=C(C=C2)C=C(N3)C4=CC=C(C=C4...,NaN,MIC,=,0.16,ug.mL-1,Bacillus_anthracis
4,72735775,C1CNC(=NC1)C2=CC3=C(C=C2)C=C(N3)CCC4=CC5=C(N4)...,NaN,MIC,=,0.63,ug.mL-1,Bacillus_anthracis
5,72736525,C1CNC(=NC1)C2=CC3=C(C=C2)C=C(N3)CCCC4=CC5=C(N4...,NaN,MIC,=,20,ug.mL-1,Bacillus_anthracis
6,72736354,C1C(CN=C(N1)C2=CC3=C(C=C2)C=C(N3)C4=CC=C(C=C4)...,NaN,MIC,=,0.08,ug.mL-1,Bacillus_anthracis
...,...,...,...,...,...,...,...,...
2005892,135907423,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC=...,0.0023,EC50,=,2.3,nM,Yersinia_pestis
2005893,73356996,CC1=CC(=CC=C1)/C(=N/NC(=O)[C@@H]2C[C@H]2C3=CC=...,0.058,EC50,=,58,nM,Yersinia_pestis
2005894,9695967,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC(...,6.3,EC50,=,6300,nM,Yersinia_pestis
2005895,25753841,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC=...,0.052,EC50,=,52,nM,Yersinia_pestis


### Print unique standard units

In [35]:
typeCountSeries = allBacteria_Pubchem["Standard Units"].dropna()
print(typeCountSeries.unique())

['ug.mL-1' 'nM' '%' 'uM' 'degrees C' 'ug ml-1' 'mm' 'hr' 'CFU' '/min' 'mM'
 'ng.hr.mL-1' 'mL.min-1.kg-1' 'L.kg-1' 'mg.kg-1' '/s' 'day' '/uM/s' 'min'
 'L/hr' 'mg/L' 'l' '/hr']


### compute `pPotency = -log10(value_in_M)` to match all the standards

In [36]:
# numeric standard value
standardValueNum = pd.to_numeric(allBacteria_Pubchem["Standard Value"], errors="coerce")

# normalize Standard Units
unitNormalized = (
    allBacteria_Pubchem["Standard Units"]
    .astype(str)
    .str.replace("\u00a0", " ", regex=False)   # NBSP
    .str.strip()
    .str.replace("µ", "u", regex=False)        # µM -> uM
    .str.replace(" ", "", regex=False)         # remove internal spaces
    .str.lower()
)

# unify known variants from your list
unitNormalized = unitNormalized.replace({
    "mm": "mm",   # keep as "mm" but interpret as mM below
})

# keep only units convertible to molar concentration
convertibleUnitsSet = {"nm", "um", "mm", "m"}
unitMask = unitNormalized.isin(convertibleUnitsSet)

allBacteria_Pubchem = allBacteria_Pubchem[unitMask].copy()
standardValueNum = standardValueNum[unitMask].reset_index(drop=True)
unitNormalized = unitNormalized[unitMask].reset_index(drop=True)

# unit -> factor to molar
unitToMolarFactorDict = {
    "nm": 1e-9,
    "um": 1e-6,
    "mm": 1e-3,  # interpret "mm" as mM
    "m": 1.0,
}

molarFactor = unitNormalized.map(unitToMolarFactorDict)
valueM = standardValueNum * molarFactor

# compute pPotency = -log10(value_in_M)
allBacteria_Pubchem["unitNormalized"] = unitNormalized
allBacteria_Pubchem["pPotency"] = np.where(valueM > 0, -np.log10(valueM), np.nan)
allBacteria_Pubchem

/users/sghosh6/.conda/envs/pubchemData/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)


,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PubChem Standard Value,Standard Type,Standard Relation,Standard Value,Standard Units,Bacteria,unitNormalized,pPotency
36,76310067,C1=CC=C(C=C1)C[C@@H](C(=O)[O-])N2C(=O)/C(=C\C3...,82,Ki,=,82000,nM,Bacillus_anthracis,nm,4.086186
118,73055013,CC1=CC(=C2C=C(C=CC2=N1)NC(=O)C3=CC=C(C=C3)C4=C...,1.14,Ki,=,1140,nM,Bacillus_anthracis,nm,5.943095
119,73055018,CC1=CC(=C2C=C(C=CC2=N1)NC(=O)CCC3=CC=CC=C3OC)N,2.88,Ki,=,2880,nM,Bacillus_anthracis,nm,5.540608
125,73055013,CC1=CC(=C2C=C(C=CC2=N1)NC(=O)C3=CC=C(C=C3)C4=C...,1.102,IC50,=,1102,nM,Bacillus_anthracis,nm,5.957818
126,73055018,CC1=CC(=C2C=C(C=CC2=N1)NC(=O)CCC3=CC=CC=C3OC)N,1.219,IC50,=,1219,nM,Bacillus_anthracis,nm,5.913996
...,...,...,...,...,...,...,...,...,...,...
2005892,135907423,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC=...,0.0023,EC50,=,2.3,nM,Yersinia_pestis,NaN,8.638272
2005893,73356996,CC1=CC(=CC=C1)/C(=N/NC(=O)[C@@H]2C[C@H]2C3=CC=...,0.058,EC50,=,58,nM,Yersinia_pestis,NaN,7.236572
2005894,9695967,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC(...,6.3,EC50,=,6300,nM,Yersinia_pestis,NaN,5.200659
2005895,25753841,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC=...,0.052,EC50,=,52,nM,Yersinia_pestis,NaN,7.283997


In [37]:
# Rename columns
allBacteria_Pubchem = allBacteria_Pubchem.rename(
    columns={'PUBCHEM_EXT_DATASOURCE_SMILES': 'Smiles', 'PUBCHEM_CID': 'compound_id'})
# Reorder columns
allBacteria_Pubchem = allBacteria_Pubchem[
    ['Bacteria', 'compound_id', 'Smiles', 'pPotency']]
allBacteria_Pubchem

,Bacteria,compound_id,Smiles,pPotency
36,Bacillus_anthracis,76310067,C1=CC=C(C=C1)C[C@@H](C(=O)[O-])N2C(=O)/C(=C\C3...,4.086186
118,Bacillus_anthracis,73055013,CC1=CC(=C2C=C(C=CC2=N1)NC(=O)C3=CC=C(C=C3)C4=C...,5.943095
119,Bacillus_anthracis,73055018,CC1=CC(=C2C=C(C=CC2=N1)NC(=O)CCC3=CC=CC=C3OC)N,5.540608
125,Bacillus_anthracis,73055013,CC1=CC(=C2C=C(C=CC2=N1)NC(=O)C3=CC=C(C=C3)C4=C...,5.957818
126,Bacillus_anthracis,73055018,CC1=CC(=C2C=C(C=CC2=N1)NC(=O)CCC3=CC=CC=C3OC)N,5.913996
...,...,...,...,...
2005892,Yersinia_pestis,135907423,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC=...,8.638272
2005893,Yersinia_pestis,73356996,CC1=CC(=CC=C1)/C(=N/NC(=O)[C@@H]2C[C@H]2C3=CC=...,7.236572
2005894,Yersinia_pestis,9695967,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC(...,5.200659
2005895,Yersinia_pestis,25753841,C/C(=N\NC(=O)[C@@H]1C[C@H]1C2=CC=CC=C2)/C3=CC=...,7.283997


### Canonicalize `SMILES`

In [38]:
def canonicalize_smiles(smi):
    mol = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(mol, canonical=True) if mol else None

allBacteria_Pubchem['Smiles'] = (
    allBacteria_Pubchem['Smiles']
    .astype(str)
    .apply(canonicalize_smiles)
)
allBacteria_Pubchem

,Bacteria,compound_id,Smiles,pPotency
36,Bacillus_anthracis,76310067,O=C([O-])[C@H](Cc1ccccc1)N1C(=O)/C(=C\c2ccc(-c...,4.086186
118,Bacillus_anthracis,73055013,Cc1cc(N)c2cc(NC(=O)c3ccc(-c4ccc5ncccc5c4)cc3)c...,5.943095
119,Bacillus_anthracis,73055018,COc1ccccc1CCC(=O)Nc1ccc2nc(C)cc(N)c2c1,5.540608
125,Bacillus_anthracis,73055013,Cc1cc(N)c2cc(NC(=O)c3ccc(-c4ccc5ncccc5c4)cc3)c...,5.957818
126,Bacillus_anthracis,73055018,COc1ccccc1CCC(=O)Nc1ccc2nc(C)cc(N)c2c1,5.913996
...,...,...,...,...
2005892,Yersinia_pestis,135907423,C/C(=N\NC(=O)[C@@H]1C[C@H]1c1ccccc1)c1ccc(O)cc1,8.638272
2005893,Yersinia_pestis,73356996,C/C(=N\NC(=O)[C@@H]1C[C@H]1c1ccccc1)c1cccc(C)c1,7.236572
2005894,Yersinia_pestis,9695967,C/C(=N\NC(=O)[C@@H]1C[C@H]1c1ccccc1)c1cccc(NC(...,5.200659
2005895,Yersinia_pestis,25753841,C/C(=N\NC(=O)[C@@H]1C[C@H]1c1ccccc1)c1ccc(F)cc1,7.283997


In [39]:
typeCountSeries = allBacteria_Pubchem["Bacteria"].dropna()

uniqueTypeCount = typeCountSeries.nunique()
totalCount = typeCountSeries.shape[0]

countDF = (
    typeCountSeries.value_counts()
    .rename_axis("Bacteria")
    .reset_index(name="count")
)
countDF["percent"] = (countDF["count"] / totalCount) * 100

print(f"Unique Bacteria type: {uniqueTypeCount}")
print(countDF)

Unique Bacteria type: 2
             Bacteria  count    percent
0  Bacillus_anthracis   3482  73.182009
1     Yersinia_pestis   1276  26.817991


In [40]:
allBacteria_Pubchem.to_csv(os.path.join(modelBuildingDataDir, "allBacteria_Pubchem.csv"), index=False)